In [3]:
# testset_ingestion.py

import os

from dotenv import load_dotenv
from langchain_core.documents import Document

# Ingestion + Summarization imports
from App.Ingestion import (
    process_pdf_input,
    table_text_segregation,
    get_images
)
from App.summarizer import summarize_texts_tables,summarize_images

# RAGAS imports
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI
from ragas.embeddings import OpenAIEmbeddings
import openai

In [4]:

# -----------------------------
# 1. Convert Summaries → LC Docs
# -----------------------------
def summaries_to_docs(text_summaries, table_summaries, img_summaries):
    """Convert all summary types into LangChain Document objects."""
    docs = []

    for s in text_summaries:
        docs.append(Document(page_content=s, metadata={"type": "text"}))

    for s in table_summaries:
        docs.append(Document(page_content=s, metadata={"type": "table"}))

    for s in img_summaries:
        docs.append(Document(page_content=s, metadata={"type": "image"}))

    return docs


In [5]:
# ------------------------------------
# 2. Ingestion Pipeline (For Testset)
# ------------------------------------
def ingestion_chain_for_testset_generation(file_path):
    """
    PDF → Extract elements → Summaries → Convert to LC Docs
    This function does NOT push anything to vector DB.
    Only produces RAGAS-ready documents.
    """
    # Extract raw elements
    elements = process_pdf_input(file_path)

    # Separate modalities
    tables, texts = table_text_segregation(elements)
    images = get_images(elements)

    # Summaries
    table_summaries, text_summaries = summarize_texts_tables(texts, tables)
    img_summaries = summarize_images(images)

    # Convert into LangChain documents
    return summaries_to_docs(text_summaries, table_summaries, img_summaries)

In [6]:
queries = [
    "How does the Transformer architecture model sequence dependencies without using recurrence or convolution?",
    "What are the main components of the encoder stack, and how do its sub-layers operate?",
    "How does the decoder use masking to preserve the autoregressive property during generation?",
    "What mathematical steps define Scaled Dot-Product Attention and why is scaling by sqrt(dk) required?",
    "What benefits does Multi-Head Attention provide compared to a single attention head?",
    "How do positional encodings represent token order, and why do sinusoidal functions enable extrapolation?",
    "What role does the position-wise Feed-Forward Network play in each Transformer layer?",
    "How does self-attention achieve a shorter maximum path length compared to RNN and CNN architectures?",
    "What computational trade-offs does self-attention introduce in terms of per-layer complexity?",
    "What data, batching strategy, and vocabulary were used for training on the WMT 2014 English-German task?",
    "How does the learning-rate schedule with warmup affect the training dynamics of the Transformer?",
    "What regularization methods (dropout, label smoothing) were applied and why?",
    "What BLEU scores did the Transformer achieve on the WMT 2014 English-German translation benchmark?",
    "How does the Transformer’s training cost compare to earlier models like GNMT or ConvS2S?",
    "What performance differences were observed when scaling from the base Transformer model to the big model?"
]


In [7]:
golden_answers = [
    # 1. How does the Transformer architecture model sequence dependencies without using recurrence or convolution?
    "In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output. The Transformer allows for significantly more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs.",

    # 2. What are the main components of the encoder stack, and how do its sub-layers operate?
    "Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-wise fully connected feed-forward network. We employ a residual connection around each of the two sub-layers, followed by layer normalization. That is, the output of each sub-layer is LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding layers, produce outputs of dimension dmodel = 512.",

    # 3. How does the decoder use masking to preserve the autoregressive property during generation?
    "We also modify the self-attention sub-layer in the decoder stack to prevent positions from attending to subsequent positions. This masking, combined with the fact that the output embeddings are offset by one position, ensures that the predictions for position i can depend only on the known outputs at positions less than i. We implement this inside of scaled dot-product attention by masking out (setting to −∞) all values in the input of the softmax which correspond to illegal connections.",

    # 4. What mathematical steps define Scaled Dot-Product Attention and why is scaling by sqrt(dk) required?
    "We compute the dot products of the query with all keys, divide each by √dk, and apply a softmax function to obtain the weights on the values. In practice, we compute the attention function on a set of queries simultaneously, packed together into a matrix Q. The keys and values are also packed together into matrices K and V. We compute the matrix of outputs as: Attention(Q, K, V ) = softmax(QKT / √dk)V. While for small values of dk the two mechanisms perform similarly, additive attention outperforms dot product attention without scaling for larger values of dk. We suspect that for large values of dk, the dot products grow large in magnitude, pushing the softmax function into regions where it has extremely small gradients. To counteract this effect, we scale the dot products by √1/dk.",

    # 5. What benefits does Multi-Head Attention provide compared to a single attention head?
    "Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions. With a single attention head, averaging inhibits this."

    # 6. How do positional encodings represent token order, and why do sinusoidal functions enable extrapolation?
    "In this work, we use sine and cosine functions of different frequencies: PE(pos,2i) = sin(pos/10000^(2i/dmodel)) and PE(pos,2i+1) = cos(pos/10000^(2i/dmodel)), where pos is the position and i is the dimension. We chose this function because we hypothesized it would allow the model to easily learn to attend by relative positions, since for any fixed offset k, PEpos+k can be represented as a linear function of PEpos. We also experimented with using learned positional embeddings instead, and found that the two versions produced nearly identical results. We chose the sinusoidal version because it may allow the model to extrapolate to sequence lengths longer than the ones encountered during training.",

    # 7. What role does the position-wise Feed-Forward Network play in each Transformer layer?
    "In addition to attention sub-layers, each of the layers in our encoder and decoder contains a fully connected feed-forward network, which is applied to each position separately and identically. This consists of two linear transformations with a ReLU activation in between. While the linear transformations are the same across different positions, they use different parameters from layer to layer.",

    # 8. How does self-attention achieve a shorter maximum path length compared to RNN and CNN architectures?
    "As noted in Table 1, a self-attention layer connects all positions with a constant number of sequentially executed operations, whereas a recurrent layer requires O(n) sequential operations. In terms of computational complexity, self-attention layers are faster than recurrent layers when the sequence length n is smaller than the representation dimensionality d. A single convolutional layer with kernel width k < n does not connect all pairs of input and output positions, and doing so requires a stack of O(n/k) convolutional layers or O(log_k(n)) in the case of dilated convolutions, increasing the length of the longest paths between any two positions in the network.",

    # 9. What computational trade-offs does self-attention introduce in terms of per-layer complexity?
    "One is the total computational complexity per layer. Another is the amount of computation that can be parallelized, as measured by the minimum number of sequential operations required. As noted in Table 1, the complexity per self-attention layer is O(n^2 · d). Convolutional layers are generally more expensive than recurrent layers, by a factor of k, though separable convolutions decrease the complexity considerably. Even with k = n, however, the complexity of a separable convolution is equal to the combination of a self-attention layer and a point-wise feed-forward layer, the approach we take in our model.",

    # 10. What data, batching strategy, and vocabulary were used for training on the WMT 2014 English-German task?
    "We trained on the standard WMT 2014 English-German dataset consisting of about 4.5 million sentence pairs. Sentences were encoded using byte-pair encoding, which has a shared source-target vocabulary of about 37000 tokens. Sentence pairs were batched together by approximate sequence length. Each training batch contained a set of sentence pairs containing approximately 25000 source tokens and 25000 target tokens."

    # 11. How does the learning-rate schedule with warmup affect the training dynamics of the Transformer?
    "We varied the learning rate over the course of training, according to the formula: lrate = d^(-0.5)_model · min(step_num^(-0.5), step_num · warmup_steps^(-1.5)). This corresponds to increasing the learning rate linearly for the first warmup_steps training steps, and decreasing it thereafter proportionally to the inverse square root of the step number.",

    # 12. What regularization methods (dropout, label smoothing) were applied and why?
    "We employ three types of regularization during training: Residual Dropout. We apply dropout to the output of each sub-layer, before it is added to the sub-layer input and normalized. In addition, we apply dropout to the sums of the embeddings and the positional encodings in both the encoder and decoder stacks. For the base model, we use a rate of P_drop = 0.1. Label Smoothing. During training, we employed label smoothing of value ε_ls = 0.1. This hurts perplexity, as the model learns to be more unsure, but improves accuracy and BLEU score.",

    # 13. What BLEU scores did the Transformer achieve on the WMT 2014 English-German translation benchmark?
    "On the WMT 2014 English-to-German translation task, the big transformer model outperforms the best previously reported models (including ensembles) by more than 2.0 BLEU, establishing a new state-of-the-art BLEU score of 28.4. Even our base model surpasses all previously published models and ensembles, at a fraction of the training cost of any of the competitive models. Table 2: The Transformer achieves better BLEU scores than previous state-of-the-art models on the English-to-German and English-to-French newstest2014 tests at a fraction of the training cost.",

    # 14. How does the Transformer’s training cost compare to earlier models like GNMT or ConvS2S?
    "Table 2 summarizes our results and compares our translation quality and training costs to other model architectures from the literature. We estimate the number of floating point operations used to train a model by multiplying the training time, the number of GPUs used, and an estimate of the sustained single-precision floating-point capacity of each GPU. The Transformer (base model) EN-DE training cost is listed as 3.3 · 10^18 FLOPs, compared to GNMT + RL at 2.3 · 10^19 FLOPs, ConvS2S at 9.6 · 10^18 FLOPs, and ByteNet, Deep-Att + PosUnk, MoE and others at significantly higher costs.",

    # 15. What performance differences were observed when scaling from the base Transformer model to the big model?
    "In Table 3 rows (C) and (D) we observe that, as expected, bigger models are better. On the WMT 2014 English-to-German translation task, the big transformer model achieves a BLEU score of 28.4, compared to the base model’s reported 27.3 in Table 2. For the big model, training took 3.5 days on 8 P100 GPUs, while the base model trained for 12 hours. The big model used d_model = 1024, d_ff = 4096, h = 16, and P_drop = 0.3, compared to the base model’s smaller configuration."
]


In [8]:
from App.console_app import initialize_retriever
from App.Ingestion_chain import ingestion_chain


# 1. Initialize retriever
ret = initialize_retriever()

# 2. Ingest your PDF into retriever
file = r"D:\Projects\Multimodal-Retrieval-Augmented-Generation\uploaded_pdfs\NIPS-2017-attention-is-all-you-need-Paper.pdf"
ingestion_chain(file, ret)

✅ Vector database initialized at: D:\Projects\Multimodal-Retrieval-Augmented-Generation\chroma_store
✅ Ollama is running.
✅ Extracted 9 elements from PDF.
📄 Texts: 9, 📊 Tables: 0, 🖼️ Images: 3
✅ Summarization complete.
✅ Added 9 text summaries.
⚠️ No table summaries to add.
✅ Added 3 image summaries.
✅ Documents added to vector database.


True

In [9]:

# 3. Build RAGAS Dataset
def build_ragas_dataset(queries, gold_answers, output="my_ragas_testset.csv"):
    from App.VectorDB import retrieve_documents
    from App.retrieval_chain import parse_docs, build_prompt
    from App.retrieval_chain import llm
    import pandas as pd




    rows = []

    for q, ref in zip(queries, gold_answers):

        # --- RETRIEVE ---
        docs = retrieve_documents(retriever=ret, question=q, k=4)

        # --- PARSE DOCS ---
        context_obj = parse_docs(docs)

        # force list of strings
        retrieved_contexts = list(map(str, context_obj["texts"] + context_obj["images"]))

        # --- BUILD PROMPT ---
        prompt_parts = build_prompt(context_obj, q)

        # --- LLM ANSWER ---
        response = llm.invoke(prompt_parts)
        answer = response.content

        # --- ADD RAGAS ROW ---
        rows.append({
            "user_input": q,
            "retrieved_contexts": retrieved_contexts,
            "response": answer,
            "reference": ref
        })

    # --- SAVE CSV ---
    df = pd.DataFrame(rows)
    df.to_csv(output, index=False)
    print(f"Saved RAGAS dataset → {output}")

    return df


In [10]:
build_ragas_dataset(queries, golden_answers, output="self_testset.csv")

Saved RAGAS dataset → self_testset.csv


,user_input,retrieved_contexts,response,reference
0,How does the Transformer architecture model se...,[Attention Is All You Need\n\nAshish Vaswani∗ ...,The Transformer architecture models sequence d...,"In this work we propose the Transformer, a mod..."
1,What are the main components of the encoder st...,[3.1 Encoder and Decoder Stacks\n\nEncoder: Th...,The encoder is composed of a stack of N = 6 id...,Encoder: The encoder is composed of a stack of...
2,How does the decoder use masking to preserve t...,[3.5 Positional Encoding\n\nSince our model co...,This document does not contain information abo...,We also modify the self-attention sub-layer in...
3,What mathematical steps define Scaled Dot-Prod...,[3.2.1 Scaled Dot-Product Attention\n\nWe call...,Scaled Dot-Product Attention is defined by the...,We compute the dot products of the query with ...
4,What benefits does Multi-Head Attention provid...,[3.2.1 Scaled Dot-Product Attention\n\nWe call...,Multi-head attention allows the model to joint...,Multi-head attention allows the model to joint...
5,How do positional encodings represent token or...,[3.5 Positional Encoding\n\nSince our model co...,Positional encodings represent token order by ...,"In addition to attention sub-layers, each of t..."
6,What role does the position-wise Feed-Forward ...,[Output Probabilities Add & Norm Feed Forward ...,The context does not explicitly state the role...,"As noted in Table 1, a self-attention layer co..."
7,How does self-attention achieve a shorter maxi...,[Output Probabilities Add & Norm Feed Forward ...,"In the Transformer, the maximum path length is...",One is the total computational complexity per ...
8,What computational trade-offs does self-attent...,[Output Probabilities Add & Norm Feed Forward ...,The context does not explicitly mention the co...,We trained on the standard WMT 2014 English-Ge...
9,"What data, batching strategy, and vocabulary w...",[5 Training\n\nThis section describes the trai...,The WMT 2014 English-German dataset consisting...,We employ three types of regularization during...


In [11]:
import pandas as pd
data = pd.read_csv("self_testset.csv")


In [12]:
import ast
data["retrieved_contexts"] = data["retrieved_contexts"].apply(ast.literal_eval)


In [13]:
from ragas import EvaluationDataset

evaluation_dataset = EvaluationDataset.from_pandas(data)


In [17]:
from ragas import evaluate
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv

load_dotenv(verbose=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Use ChatOpenAI directly - RAGAS supports it natively
evaluator_llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY,
    temperature=0  # Optional: for more consistent evaluation
)

from ragas.metrics import (
    LLMContextRecall,
    Faithfulness,
    FactualCorrectness,
    ContextPrecision,
    AnswerRelevancy,
    SemanticSimilarity
)

metrics = [
    LLMContextRecall(),
    ContextPrecision(),
    Faithfulness(),
    FactualCorrectness(),
    AnswerRelevancy(),
    SemanticSimilarity(),
]

result = evaluate(
    dataset=evaluation_dataset,
    metrics=metrics,
    llm=evaluator_llm,
)

Evaluating:   0%|          | 0/78 [00:00<?, ?it/s]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:   6%|▋         | 5/78 [00:49<09:54,  8.14s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  49%|████▊     | 38/78 [04:21<06:56, 10.41s/it]Exception raised in Job[25]: TimeoutError()
Exception raised in Job[19]: TimeoutError()
Evaluating: 100%|██████████| 78/78 [07:44<00:00,  5.95s/it]


In [18]:
results=result.to_pandas()

results.to_csv("result.csv",index=False)
print("Saved metrics → ragas_metrics_results.csv")


Saved metrics → ragas_metrics_results.csv


In [20]:
results.head()

,user_input,retrieved_contexts,response,reference,context_recall,context_precision,faithfulness,factual_correctness(mode=f1),answer_relevancy,semantic_similarity
0,How does the Transformer architecture model se...,[Attention Is All You Need\n\nAshish Vaswani∗ ...,The Transformer architecture models sequence d...,"In this work we propose the Transformer, a mod...",1.000000,0.805556,1.0,0.55,0.947827,0.918039
1,What are the main components of the encoder st...,[3.1 Encoder and Decoder Stacks\n\nEncoder: Th...,The encoder is composed of a stack of N = 6 id...,Encoder: The encoder is composed of a stack of...,1.000000,1.000000,1.0,0.82,0.901172,0.982003
2,How does the decoder use masking to preserve t...,[3.5 Positional Encoding\n\nSince our model co...,This document does not contain information abo...,We also modify the self-attention sub-layer in...,1.000000,NaN,0.0,0.00,0.000000,0.778220
3,What mathematical steps define Scaled Dot-Prod...,[3.2.1 Scaled Dot-Product Attention\n\nWe call...,Scaled Dot-Product Attention is defined by the...,We compute the dot products of the query with ...,1.000000,NaN,0.9,0.76,0.902181,0.941701
4,What benefits does Multi-Head Attention provid...,[3.2.1 Scaled Dot-Product Attention\n\nWe call...,Multi-head attention allows the model to joint...,Multi-head attention allows the model to joint...,0.333333,NaN,1.0,0.31,0.928391,0.902928
